# EV hypothesis playground

This notebook uses the same `Service` boundary as the dashboard. It creates a synthetic planning experiment, waits for a run, and inspects returned results without assuming a fixed metric set. The model is not an as-built grid and vehicle sessions are hypotheses, not observed travel.

In [ ]:
from pathlib import Path
import sys

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'mvgrid').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from inside the repository checkout.')
sys.path.insert(0, str(root / 'src'))

from mvgrid.novi_sad.playground.service import Service
service = Service()


## Inspect and validate the bundled example

Edit a copy to change demand, fleet, charging strategy, district capacities or district placement. The simulation uses 15-minute intervals. `validate_experiment` is the authoritative schema check.


In [ ]:
from copy import deepcopy
from pprint import pprint

definition = deepcopy(service.catalog()['example'])
validation = service.validate_experiment(definition)
pprint(validation)
if validation.get('valid') is False or validation.get('errors'):
    raise ValueError(validation.get('errors', 'Invalid experiment'))


## Save, start, and poll

Runs are persisted by the service. The polling loop has a timeout and can be interrupted safely; use `cancel_run(run_id)` to request cancellation or call `start_run(experiment_id, resume_run_id=run_id)` to resume an interrupted run.

In [ ]:
import time

saved = service.save_experiment(definition)
experiment_id = saved['experiment_id']
started = service.start_run(experiment_id)
run_id = started['run_id']

terminal = {'completed', 'failed', 'cancelled', 'interrupted', 'rejected', 'budget_exceeded', 'stopped_on_violation'}
deadline = time.monotonic() + 300
while True:
    run = service.get_run(run_id)
    print(run_id, run.get('status'))
    if str(run.get('status', '')).lower() in terminal:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError('Run did not finish within five minutes; it remains available through list_runs().')
    time.sleep(1)

run


## Explore returned cases

The service returns case records with `metrics`, `intervals`, `blocks`, and `sessions`. Tables below keep all available fields, so new engine metrics remain visible.

In [ ]:
import pandas as pd

results = service.get_results(run_id)
cases = results.get('cases', [])
case_summary = pd.DataFrame([
    {'case_id': c.get('case_id'), 'strategy': c.get('strategy'), 'seed': c.get('seed'),
     'fleet_size': c.get('fleet_size'), **c.get('metrics', {})}
    for c in cases
])
case_summary


In [ ]:
case = cases[0]
intervals = pd.DataFrame(case.get('intervals', []))
sessions = pd.DataFrame(case.get('sessions', []))
display(intervals.head(), sessions.head())

from matplotlib.ticker import FuncFormatter

demand_columns = [c for c in ['baseline_kw', 'ev_kw', 'total_kw'] if c in intervals]
if demand_columns:
    ax = intervals[demand_columns].plot(figsize=(11, 4), title=f"Demand - {case.get('case_id')}", ylabel='kW')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda step, _: f"D{int(step)//96+1} {int(step)%96//4:02d}:{int(step)%4*15:02d}"))
    ax.set_xlabel('Time (HH:MM)')
    ax.set_xlim(0, max(1, len(intervals)-1))


## Frozen district capacity and car placement

No individual MV/LV assets are modeled. District budgets are planning estimates, and their provenance and scenario factors belong to the saved experiment. Historical runs without these records did not evaluate district-capacity constraints.


In [ ]:
display(pd.DataFrame(case.get("resolved_districts", [])))
if len(intervals):
    display(pd.DataFrame(intervals.iloc[0].get("districts", [])))
if "district_id" in sessions.columns:
    display(sessions.groupby("district_id").size().rename("session_count"))


## Compare persisted runs

Pass two or more run IDs to the service to obtain its canonical comparison rather than calculating a separate notebook metric.

In [ ]:
available_run_ids = [r.get('run_id', r.get('id')) for r in service.list_runs()]
if len(available_run_ids) >= 2:
    comparison = service.compare_runs(available_run_ids[:2])
    display(pd.DataFrame(comparison.get('rows', comparison)) if isinstance(comparison, dict) else pd.DataFrame(comparison))
else:
    print('Create another run to compare scenarios.')
